<!-- artqr-doc -->
# ArtQR Fusion Notebook

Bu notebook, yuklenen bir gorseli okunabilir QR kod yapisiyla birlestirir. Amac, klasik siyah-beyaz QR yerine marka, logo veya sanat gorseliyle daha estetik ama taranabilir bir QR uretmektir.


<!-- artqr-doc -->
## 1. Ortam Kurulumu

Colab ortaminda gerekli sistem paketleri ve Python kutuphaneleri kurulur. QR okuma icin `libzbar0`, gorsel isleme icin Pillow/OpenCV, QR uretimi icin `segno`, dogrulama icin `pyzbar` hazirlanir.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y libzbar0
!pip -q install --upgrade "Pillow<12" opencv-python pyzbar segno numpy zxing-cpp


<!-- artqr-doc -->
## 2. Kutuphaneleri Yukleme

Notebook boyunca kullanilacak tum kutuphaneler tek yerde ice aktarilir. Bu hucre dosya yukleme, gorsel isleme, QR uretme, QR okuma ve ciktilari gostermek icin temel araclari hazirlar.


In [ ]:
import os
import cv2
import math
import zipfile
import numpy as np
import segno
from PIL import Image, ImageEnhance, ImageFilter, ImageDraw, ImageOps
from pyzbar.pyzbar import decode as zbar_decode
from IPython.display import display
from google.colab import files

try:
    import zxingcpp
except Exception as exc:
    zxingcpp = None
    print("ZXing yuklenemedi; pyzbar + OpenCV ile devam edilecek:", exc)


<!-- artqr-doc -->
## 3. Gorsel Yukleme

QR kodun icine islenecek ana gorsel bu hucrede yuklenir. Yuklenen dosya onizleme olarak gosterilir; boylece devam etmeden once dogru gorselin secildigini kontrol edebilirsiniz.


In [ ]:
#@title 3. Gorsel ve QR Giris Formu
# Gorsel ve QR giris modu
# upload_mode="generate_qr_from_data": QR, qr_data alanindan Segno ile uretilir.
# upload_mode="use_uploaded_qr_image": Elinizdeki hazir qr_code.png yuklenir ve fusion maskesi olarak kullanilir.
upload_mode = "generate_qr_from_data" #@param ["generate_qr_from_data", "use_uploaded_qr_image"]
upload_artwork_image = True #@param {type:"boolean"}
upload_qr_image_when_needed = True #@param {type:"boolean"}

artwork_path = ""
uploaded_qr_path = ""

if upload_artwork_image:
    print("Logo / artwork gorselini yukleyin.")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Artwork gorseli yuklenmedi.")
    artwork_path = list(uploaded.keys())[0]
else:
    artwork_path = "artwork.png" #@param {type:"string"}

if upload_mode == "use_uploaded_qr_image" and upload_qr_image_when_needed:
    print("Hazir QR gorselini yukleyin.")
    uploaded_qr = files.upload()
    if not uploaded_qr:
        raise ValueError("QR gorseli yuklenmedi.")
    uploaded_qr_path = list(uploaded_qr.keys())[0]
elif upload_mode == "use_uploaded_qr_image":
    uploaded_qr_path = "qr_code.png" #@param {type:"string"}

print("Yuklenen artwork:", artwork_path)
if uploaded_qr_path:
    print("Yuklenen QR:", uploaded_qr_path)

artwork_preview = Image.open(artwork_path).convert("RGBA")
display(artwork_preview)


<!-- artqr-doc -->
## 4. Ana QR ve Gorsel Ayarlari

QR icine yazilacak veri, cikti boyutu, hata toleransi, kenar boslugu ve gorselin kare alana nasil oturtulacagi burada belirlenir. Ayrica QR deseninin gorselle ne kadar guclu karisacagini ayarlayan kontrast, keskinlik ve okunabilirlik parametreleri de bu bolumdedir.


In [ ]:
# Ana QR ve fusion ayarlari
# qr_data: Segno ile QR uretilirken kullanilir. Hazir QR yukleme modunda beklenen okuma degeri olarak da kullanilabilir.
# expected_qr_data bos birakilirsa qr_data beklenen deger kabul edilir.
qr_data = "https://example.com" #@param {type:"string"}
expected_qr_data = "" #@param {type:"string"}
output_size = 3072 #@param {type:"slider", min:1024, max:4096, step:512}
error_level = "h" #@param ["h", "q", "m", "l"]
quiet_zone = 4 #@param {type:"slider", min:4, max:10, step:1}

fit_mode = "cover_crop" #@param ["cover_crop", "contain_pad"]
pad_red = 255 #@param {type:"slider", min:0, max:255, step:1}
pad_green = 255 #@param {type:"slider", min:0, max:255, step:1}
pad_blue = 255 #@param {type:"slider", min:0, max:255, step:1}

# Fusion araligi: okunabilirlik zayifsa end/contrast/finder artirilir; gorsel sertlesirse start/contrast dusurulur.
dark_strength_start = 0.25 #@param {type:"slider", min:0.05, max:0.80, step:0.01}
dark_strength_end = 0.85 #@param {type:"slider", min:0.10, max:0.95, step:0.01}
dark_strength_step = 0.04 #@param {type:"slider", min:0.01, max:0.10, step:0.01}
light_strength_ratio = 0.8 #@param {type:"slider", min:0.20, max:1.00, step:0.05}
module_contrast = 0.7 #@param {type:"slider", min:0.00, max:1.00, step:0.05}
center_bias = 0.8 #@param {type:"slider", min:0.00, max:1.00, step:0.05}

finder_boost = 2 #@param {type:"slider", min:1.00, max:2.50, step:0.05}
quiet_zone_boost = 0.7 #@param {type:"slider", min:0.00, max:1.00, step:0.05}
contrast_after = 1.12 #@param {type:"slider", min:1.00, max:1.80, step:0.02}
sharpness_after = 1.35 #@param {type:"slider", min:1.00, max:2.50, step:0.05}

# Ticari kabul kuralı: pyzbar, OpenCV, ZXing icinden en az bu kadar okuyucu dogru sonucu okumali.
min_decoder_passes = 2 #@param {type:"slider", min:1, max:3, step:1}
prefer_best_score_not_first_pass = True #@param {type:"boolean"}

save_all_candidates = True #@param {type:"boolean"}
show_candidate_previews = True #@param {type:"boolean"}
download_final = True #@param {type:"boolean"}

expected_data = expected_qr_data.strip() or qr_data


<!-- artqr-doc -->
## 5. Ticari Stil ve Logo Ayarlari

Daha profesyonel teslim dosyalari icin logo, rozet, modul sekli, renk paleti, arka plan ve dosya formatlari burada ayarlanir. Bu bolum, musterilere sunulabilecek daha temiz ve markali QR varyasyonlari uretmek icin kullanilir.


In [ ]:
#@title 5. Ticari Stil, Logo ve QR Sekil Formu
# Ticari stil ve teslim ayarlari
# style_preset secimi alttaki marka/renk/sekil ayarlarini tek hamlede duzenler; custom secilirse elle girilen degerler korunur.
commercial_pack_enabled = True #@param {type:"boolean"}
style_preset = "business_clean" #@param ["custom", "business_clean", "luxury", "colorful", "restaurant", "event", "product_label", "social_media"]

logo_source_mode = "uploaded_artwork" #@param ["uploaded_artwork", "none"]
logo_mode = "center_badge" #@param ["none", "center_badge", "center_transparent", "background_logo", "corner_small", "logo_under_qr", "qr_over_logo_soft"]
logo_size_ratio = 0.16 #@param {type:"slider", min:0.08, max:0.26, step:0.01}
logo_badge_shape = "rounded_square" #@param ["rounded_square", "circle", "square"]
logo_badge_opacity = 0.96 #@param {type:"slider", min:0.40, max:1.00, step:0.05}
logo_badge_padding = 0.28 #@param {type:"slider", min:0.05, max:0.45, step:0.01}
logo_visual_strength = 0.90 #@param {type:"slider", min:0.20, max:1.00, step:0.02}
logo_auto_readability_guard = True #@param {type:"boolean"}
brand_text = "" #@param {type:"string"}
logo_mode_from_preset = False #@param {type:"boolean"}

# Logo kalite hazirligi: real_esrgan modu hazirlik anahtari olarak durur; ortamda model yoksa OpenCV/Pillow upscale fallback calisir.
logo_prep_enabled = True #@param {type:"boolean"}
logo_upscale_engine = "opencv_lanczos" #@param ["none", "opencv_lanczos", "real_esrgan"]
logo_upscale_factor = 2 #@param {type:"slider", min:1, max:4, step:1}
logo_denoise = "light" #@param ["none", "light", "medium"]
logo_sharpen = "medium" #@param ["none", "light", "medium", "strong"]

module_shape = "rounded" #@param ["square", "rounded", "dot", "diamond", "hexagon", "vertical_line", "horizontal_line", "soft_blob"]
finder_style = "rounded" #@param ["classic", "rounded", "circle"]
color_mode = "brand_color" #@param ["black_white", "brand_color", "two_color", "gradient", "image_based"]
dark_hex = "#111111" #@param {type:"string"}
brand_hex = "#1F6FEB" #@param {type:"string"}
gradient_start_hex = "#111111" #@param {type:"string"}
gradient_end_hex = "#1F6FEB" #@param {type:"string"}
light_hex = "#FFFFFF" #@param {type:"string"}

background_style = "white" #@param ["white", "brand_tint", "transparent", "artwork"]
brand_tint_strength = 0.10 #@param {type:"slider", min:0.00, max:0.35, step:0.01}

make_soft_balanced_strong = True #@param {type:"boolean"}
make_styled_clean_qr = True #@param {type:"boolean"}
make_halftone_logo_qr = True #@param {type:"boolean"}
halftone_mark_shape = "plus" #@param ["dot", "square", "plus", "diamond", "star"]
halftone_density = 0.88 #@param {type:"slider", min:0.40, max:1.00, step:0.02}
halftone_logo_strength = 0.78 #@param {type:"slider", min:0.20, max:1.00, step:0.02}
halftone_background_dots = True #@param {type:"boolean"}
halftone_designer_finders = True #@param {type:"boolean"}
make_social_preview = True #@param {type:"boolean"}
make_print_png = True #@param {type:"boolean"}
make_print_pdf = True #@param {type:"boolean"}
make_transparent_png = True #@param {type:"boolean"}
make_jpg = True #@param {type:"boolean"}
make_svg = True #@param {type:"boolean"}
make_eps = False #@param {type:"boolean"}
make_layout_pack = True #@param {type:"boolean"}
make_test_report = True #@param {type:"boolean"}

PRESETS = {
    "business_clean": dict(module_shape="rounded", finder_style="rounded", color_mode="brand_color", brand_hex="#1F6FEB", background_style="white", logo_mode="center_badge"),
    "luxury": dict(module_shape="rounded", finder_style="rounded", color_mode="gradient", gradient_start_hex="#151515", gradient_end_hex="#B08D57", background_style="brand_tint", brand_hex="#B08D57", logo_mode="center_badge"),
    "colorful": dict(module_shape="soft_blob", finder_style="circle", color_mode="gradient", gradient_start_hex="#FF4D6D", gradient_end_hex="#1FDDFF", background_style="white", logo_mode="center_transparent"),
    "restaurant": dict(module_shape="rounded", finder_style="classic", color_mode="two_color", dark_hex="#1F1A17", brand_hex="#D1495B", background_style="brand_tint", logo_mode="center_badge"),
    "event": dict(module_shape="dot", finder_style="circle", color_mode="gradient", gradient_start_hex="#111111", gradient_end_hex="#7C3AED", background_style="white", logo_mode="corner_small"),
    "product_label": dict(module_shape="square", finder_style="classic", color_mode="black_white", background_style="white", logo_mode="logo_under_qr"),
    "social_media": dict(module_shape="rounded", finder_style="rounded", color_mode="image_based", background_style="artwork", logo_mode="qr_over_logo_soft"),
}

if style_preset != "custom":
    selected_logo_mode = logo_mode
    for key, value in PRESETS[style_preset].items():
        if key == "logo_mode" and not logo_mode_from_preset:
            continue
        globals()[key] = value
    if not logo_mode_from_preset:
        logo_mode = selected_logo_mode
    print("Preset uygulandi:", style_preset, "| aktif logo_mode:", logo_mode)


<!-- artqr-doc -->
## 6. Gorsel Hazirlama ve QR Test Fonksiyonlari

Bu yardimci fonksiyonlar yuklenen gorseli kare formata getirir, temiz QR kodu olusturur ve uretilen dosyalarin gercekten okunup okunmadigini test eder. Notebook'un guvenli calismasi icin temel altyapi bu hucrededir.


In [ ]:
def prepare_artwork(path, size, fit_mode="cover_crop", pad_color=(255, 255, 255)):
    img = Image.open(path).convert("RGB")
    w, h = img.size

    if fit_mode == "cover_crop":
        scale = size / min(w, h)
        nw, nh = int(round(w * scale)), int(round(h * scale))
        img = img.resize((nw, nh), Image.Resampling.LANCZOS)
        left = (nw - size) // 2
        top = (nh - size) // 2
        img = img.crop((left, top, left + size, top + size))
        return img

    scale = size / max(w, h)
    nw, nh = int(round(w * scale)), int(round(h * scale))
    img = img.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (size, size), pad_color)
    canvas.paste(img, ((size - nw) // 2, (size - nh) // 2))
    return canvas

def make_qr_png(data, path, size, error="h", border=4):
    qr = segno.make(data, error=error, boost_error=True)
    modules_with_border = qr.symbol_size(scale=1, border=border)[0]
    scale = max(1, size // modules_with_border)
    render_size = modules_with_border * scale
    qr.save(path, kind="png", scale=scale, border=border, dark="black", light="white")
    img = Image.open(path).convert("L")
    if img.size != (size, size):
        img = img.resize((size, size), Image.Resampling.NEAREST)
        img.save(path)
    return qr, img

def get_qr_masks(qr_img):
    arr = np.array(qr_img.convert("L"))
    dark_mask = arr < 128
    light_mask = ~dark_mask
    return dark_mask, light_mask

def build_finder_mask(size, qr, border):
    if qr is None:
        return np.zeros((size, size), dtype=bool)
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    qr_size_no_border = modules_total - 2 * border

    finder_mask = np.zeros((size, size), dtype=bool)

    # Finder pattern: QR okuyucunun once buldugu 3 buyuk kose karesi.
    # 7 modul ana kare + 1 modul separator bolgesi dahil edilerek guclendirilir.
    zones = [
        (border - 1, border - 1),
        (border + qr_size_no_border - 8, border - 1),
        (border - 1, border + qr_size_no_border - 8),
    ]

    for mx, my in zones:
        x1 = max(0, int(math.floor(mx * module_px)))
        y1 = max(0, int(math.floor(my * module_px)))
        x2 = min(size, int(math.ceil((mx + 9) * module_px)))
        y2 = min(size, int(math.ceil((my + 9) * module_px)))
        finder_mask[y1:y2, x1:x2] = True

    return finder_mask

def build_quiet_zone_mask(size, qr, border):
    if qr is None:
        b = max(1, int(round(size * 0.06)))
        mask = np.zeros((size, size), dtype=bool)
        mask[:b, :] = True
        mask[-b:, :] = True
        mask[:, :b] = True
        mask[:, -b:] = True
        return mask
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    b = int(round(border * module_px))
    mask = np.zeros((size, size), dtype=bool)
    mask[:b, :] = True
    mask[-b:, :] = True
    mask[:, :b] = True
    mask[:, -b:] = True
    return mask

def build_module_center_weight(size, qr, border, center_bias):
    if qr is None or center_bias <= 0:
        return np.ones((size, size), dtype=np.float32)

    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total

    yy, xx = np.indices((size, size))
    mx = (xx / module_px) % 1.0
    my = (yy / module_px) % 1.0

    dx = np.abs(mx - 0.5) * 2.0
    dy = np.abs(my - 0.5) * 2.0
    dist = np.maximum(dx, dy)
    center = np.clip(1.0 - dist, 0.0, 1.0)
    return (1.0 - center_bias) + center_bias * center

def luminance_fusion(
    art_img,
    qr_img,
    qr,
    border,
    dark_strength,
    light_ratio,
    finder_boost,
    quiet_boost,
    module_contrast=0.35,
    center_bias=0.65,
):
    art = np.array(art_img.convert("RGB"))
    dark_mask, light_mask = get_qr_masks(qr_img)
    finder_mask = build_finder_mask(art.shape[0], qr, border)
    quiet_mask = build_quiet_zone_mask(art.shape[0], qr, border)
    center_weight = build_module_center_weight(art.shape[0], qr, border, center_bias)

    lab = cv2.cvtColor(art, cv2.COLOR_RGB2LAB).astype(np.float32)
    l = lab[:, :, 0]

    strength_map = np.full(l.shape, dark_strength, dtype=np.float32)
    strength_map *= center_weight
    strength_map[finder_mask] *= finder_boost
    strength_map = np.clip(strength_map, 0.0, 0.95)

    dark_floor = 255.0 * (1.0 - module_contrast)
    light_floor = 255.0 * module_contrast

    dark_target = np.minimum(l * (1.0 - strength_map), dark_floor)
    light_strength = np.clip(strength_map * light_ratio, 0.0, 0.90)
    light_target = np.maximum(l + (255.0 - l) * light_strength, light_floor)

    l2 = l.copy()
    l2[dark_mask] = dark_target[dark_mask]
    l2[light_mask] = light_target[light_mask]

    if quiet_boost > 0:
        l2[quiet_mask] = l2[quiet_mask] + (255.0 - l2[quiet_mask]) * quiet_boost

    lab[:, :, 0] = np.clip(l2, 0, 255)
    fused = cv2.cvtColor(lab.astype(np.uint8), cv2.COLOR_LAB2RGB)
    return Image.fromarray(fused)

def polish_image(img, contrast=1.08, sharpness=1.25):
    img = ImageEnhance.Contrast(img).enhance(contrast)
    img = ImageEnhance.Sharpness(img).enhance(sharpness)
    img = img.filter(ImageFilter.UnsharpMask(radius=1.0, percent=90, threshold=3))
    return img


def analyze_logo_quality(path):
    img = Image.open(path).convert("RGBA")
    rgb = img.convert("RGB")
    arr = np.array(rgb)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    alpha = np.array(img.getchannel("A"))
    warnings = []
    w, h = img.size
    if min(w, h) < 600:
        warnings.append("Cozunurluk dusuk; logo_upscale_factor 2-4 onerilir.")
    if alpha.min() < 255:
        transparent_ratio = float((alpha < 250).mean())
        if transparent_ratio > 0.10:
            warnings.append("Transparan alan yuksek; badge veya beyaz zemin daha guvenli olabilir.")
    contrast = float(gray.std())
    if contrast < 32:
        warnings.append("Kontrast zayif; module_contrast/finder_boost veya sharpen artirilabilir.")
    edges = cv2.Laplacian(gray, cv2.CV_64F).var()
    if edges < 80:
        warnings.append("Gorsel yumusak/bulanik; sharpen medium/strong onerilir.")
    small_detail_ratio = float((cv2.Canny(gray, 80, 160) > 0).mean())
    if small_detail_ratio > 0.18:
        warnings.append("Cok ince detay/kucuk yazi var; QR icinde kaybolabilir.")
    return {
        "size": img.size,
        "contrast": round(contrast, 2),
        "sharpness": round(float(edges), 2),
        "transparent_ratio": round(float((alpha < 250).mean()), 3),
        "small_detail_ratio": round(small_detail_ratio, 3),
        "warnings": warnings,
    }

def prepare_logo_source(path, out_path="qr_fusion_outputs/prepared_logo_source.png"):
    img = Image.open(path).convert("RGBA")
    if not logo_prep_enabled:
        img.save(out_path)
        return out_path
    if logo_upscale_engine in ("opencv_lanczos", "real_esrgan") and logo_upscale_factor > 1:
        if logo_upscale_engine == "real_esrgan":
            print("Real-ESRGAN model entegrasyonu bu notebookta opsiyonel hazirlik anahtari olarak duruyor; OpenCV Lanczos fallback kullaniliyor.")
        new_size = (img.size[0] * int(logo_upscale_factor), img.size[1] * int(logo_upscale_factor))
        img = img.resize(new_size, Image.Resampling.LANCZOS)
    if logo_denoise != "none":
        strength = 3 if logo_denoise == "light" else 7
        arr = cv2.cvtColor(np.array(img.convert("RGB")), cv2.COLOR_RGB2BGR)
        arr = cv2.fastNlMeansDenoisingColored(arr, None, strength, strength, 7, 21)
        rgb = Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)).convert("RGBA")
        rgb.putalpha(img.getchannel("A"))
        img = rgb
    sharpen_map = {"none": 1.0, "light": 1.15, "medium": 1.35, "strong": 1.65}
    if sharpen_map.get(logo_sharpen, 1.0) > 1.0:
        rgb = ImageEnhance.Sharpness(img.convert("RGB")).enhance(sharpen_map[logo_sharpen])
        rgb = rgb.filter(ImageFilter.UnsharpMask(radius=1.0, percent=110, threshold=3))
        rgb = rgb.convert("RGBA")
        rgb.putalpha(img.getchannel("A"))
        img = rgb
    img.save(out_path)
    return out_path

def load_uploaded_qr_image(path, size):
    img = Image.open(path).convert("L")
    img = ImageOps.contain(img, (size, size), Image.Resampling.LANCZOS)
    canvas = Image.new("L", (size, size), 255)
    canvas.paste(img, ((size - img.size[0]) // 2, (size - img.size[1]) // 2))
    canvas = canvas.point(lambda p: 0 if p < 160 else 255)
    return canvas

def test_qr_all(path, expected_data=None):
    pil_img = Image.open(path).convert("RGB")

    zbar_results = zbar_decode(pil_img)
    zbar_texts = [r.data.decode("utf-8") for r in zbar_results]

    cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    detector = cv2.QRCodeDetector()
    cv_text, points, _ = detector.detectAndDecode(cv_img)
    cv_texts = [cv_text] if cv_text else []

    zxing_texts = []
    if zxingcpp is not None:
        try:
            zxing_results = zxingcpp.read_barcodes(pil_img)
            zxing_texts = [r.text for r in zxing_results if getattr(r, "text", None)]
        except Exception:
            zxing_texts = []

    engines = {"zbar": zbar_texts, "opencv": cv_texts, "zxing": zxing_texts}
    all_texts = list(dict.fromkeys(zbar_texts + cv_texts + zxing_texts))
    if expected_data:
        passed_engines = [name for name, texts in engines.items() if expected_data in texts]
    else:
        passed_engines = [name for name, texts in engines.items() if texts]
    ok = len(passed_engines) >= min_decoder_passes

    return {
        "ok": ok,
        "texts": all_texts,
        "passed_engines": passed_engines,
        "decoder_pass_count": len(passed_engines),
        "zbar": zbar_texts,
        "opencv": cv_texts,
        "zxing": zxing_texts,
    }

def qr_quality_score(path, result, strength=0.0):
    img = Image.open(path).convert("L")
    arr = np.array(img, dtype=np.float32)
    contrast = float(arr.std() / 64.0)
    readability = float(result.get("decoder_pass_count", 0)) * 10.0
    aesthetic_penalty = abs(float(strength) - 0.55) * 2.0
    return readability + min(contrast, 4.0) - aesthetic_penalty


<!-- artqr-doc -->
## 7. Stil Cizim Fonksiyonlari

Bu bolum renkleri donusturur, modul sekillerini cizer, finder alanlarini duzenler, logo/rozet ekler ve ticari gorunumleri olusturur. Yani QR kodun sadece okunur degil, ayni zamanda guzel gorunmesini saglayan gorsel motor burada calisir.


In [ ]:
def hex_to_rgb(value):
    value = value.strip().lstrip("#")
    if len(value) == 3:
        value = "".join(ch * 2 for ch in value)
    if len(value) != 6:
        raise ValueError(f"Gecersiz hex renk: {value}")
    return tuple(int(value[i:i + 2], 16) for i in (0, 2, 4))

def blend_rgb(a, b, t):
    return tuple(int(round(a[i] * (1.0 - t) + b[i] * t)) for i in range(3))

def make_background(size, style, light_rgb, brand_rgb, artwork=None, transparent=False):
    if transparent:
        return Image.new("RGBA", (size, size), (255, 255, 255, 0))
    if style == "artwork" and artwork is not None:
        return artwork.convert("RGBA")
    if style == "brand_tint":
        rgb = blend_rgb(light_rgb, brand_rgb, brand_tint_strength)
        return Image.new("RGBA", (size, size), rgb + (255,))
    return Image.new("RGBA", (size, size), light_rgb + (255,))

def module_is_dark(qr_img, x1, y1, x2, y2):
    cx = min(qr_img.size[0] - 1, max(0, int(round((x1 + x2) / 2))))
    cy = min(qr_img.size[1] - 1, max(0, int(round((y1 + y2) / 2))))
    return qr_img.getpixel((cx, cy)) < 128

def draw_module(draw, box, shape, fill, radius_ratio=0.35):
    x1, y1, x2, y2 = box
    inset = max(1, int(round((x2 - x1) * 0.10)))
    box = (x1 + inset, y1 + inset, x2 - inset, y2 - inset)
    if shape == "dot":
        draw.ellipse(box, fill=fill)
    elif shape == "diamond":
        cx = (box[0] + box[2]) / 2
        cy = (box[1] + box[3]) / 2
        draw.polygon([(cx, box[1]), (box[2], cy), (cx, box[3]), (box[0], cy)], fill=fill)
    elif shape == "hexagon":
        w = box[2] - box[0]
        h = box[3] - box[1]
        draw.polygon([(box[0] + w*0.25, box[1]), (box[0] + w*0.75, box[1]), (box[2], box[1] + h*0.5), (box[0] + w*0.75, box[3]), (box[0] + w*0.25, box[3]), (box[0], box[1] + h*0.5)], fill=fill)
    elif shape == "vertical_line":
        cx = (box[0] + box[2]) / 2
        line_w = max(1, int((box[2] - box[0]) * 0.42))
        draw.rounded_rectangle((cx - line_w/2, box[1], cx + line_w/2, box[3]), radius=max(1, line_w//2), fill=fill)
    elif shape == "horizontal_line":
        cy = (box[1] + box[3]) / 2
        line_h = max(1, int((box[3] - box[1]) * 0.42))
        draw.rounded_rectangle((box[0], cy - line_h/2, box[2], cy + line_h/2), radius=max(1, line_h//2), fill=fill)
    elif shape == "soft_blob":
        radius = max(1, int(round((box[2] - box[0]) * 0.48)))
        draw.rounded_rectangle(box, radius=radius, fill=fill)
    elif shape == "rounded":
        radius = max(1, int(round((box[2] - box[0]) * radius_ratio)))
        draw.rounded_rectangle(box, radius=radius, fill=fill)
    else:
        draw.rectangle(box, fill=fill)


def make_logo_mask(logo_path, size, threshold=245):
    logo = Image.open(logo_path).convert("RGBA")
    fitted = ImageOps.contain(logo, (int(size * 0.90), int(size * 0.90)), Image.Resampling.LANCZOS)
    layer = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    layer.paste(fitted, ((size - fitted.size[0]) // 2, (size - fitted.size[1]) // 2), fitted)
    alpha = np.array(layer.getchannel("A"))
    rgb = np.array(layer.convert("RGB"))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    color_delta = np.max(np.abs(rgb.astype(np.int16) - 255), axis=2)
    mask = (alpha > 24) & ((gray < threshold) | (color_delta > 18))
    mask = cv2.morphologyEx(mask.astype(np.uint8) * 255, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8)) > 0
    return layer, mask

def logo_sample_color(logo_layer, x1, y1, x2, y2, fallback):
    crop = logo_layer.crop((x1, y1, max(x1 + 1, x2), max(y1 + 1, y2))).convert("RGBA")
    arr = np.array(crop)
    alpha = arr[:, :, 3] > 24
    if not alpha.any():
        return fallback
    rgb = arr[:, :, :3][alpha]
    color = tuple(int(v) for v in np.median(rgb, axis=0))
    return color + (255,)

def draw_halftone_mark(draw, box, shape, fill, density=0.86):
    x1, y1, x2, y2 = box
    w = max(1, x2 - x1)
    h = max(1, y2 - y1)
    inset = int(round(min(w, h) * (1.0 - density) * 0.45))
    x1 += inset; y1 += inset; x2 -= inset; y2 -= inset
    if x2 <= x1 or y2 <= y1:
        return
    if shape == "dot":
        draw.ellipse((x1, y1, x2, y2), fill=fill)
    elif shape == "square":
        draw.rectangle((x1, y1, x2, y2), fill=fill)
    elif shape == "diamond":
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        draw.polygon([(cx, y1), (x2, cy), (cx, y2), (x1, cy)], fill=fill)
    elif shape == "star":
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        rx = (x2 - x1) * 0.48
        ry = (y2 - y1) * 0.48
        pts = []
        for i in range(8):
            r = 1.0 if i % 2 == 0 else 0.42
            a = -math.pi / 2 + i * math.pi / 4
            pts.append((cx + math.cos(a) * rx * r, cy + math.sin(a) * ry * r))
        draw.polygon(pts, fill=fill)
    else:
        cx = (x1 + x2) / 2
        cy = (y1 + y2) / 2
        arm = max(1, int(round(min(x2 - x1, y2 - y1) * 0.28)))
        draw.rounded_rectangle((cx - arm / 2, y1, cx + arm / 2, y2), radius=max(1, arm // 2), fill=fill)
        draw.rounded_rectangle((x1, cy - arm / 2, x2, cy + arm / 2), radius=max(1, arm // 2), fill=fill)

def draw_designer_finder(draw, box, accent, light=(255, 255, 255, 255), dark=(18, 18, 18, 255)):
    x1, y1, x2, y2 = box
    pad = max(2, int(round((x2 - x1) * 0.05)))
    outer = (x1 + pad, y1 + pad, x2 - pad, y2 - pad)
    draw.ellipse(outer, fill=dark)
    ring_pad = max(2, int(round((x2 - x1) * 0.13)))
    ring = (outer[0] + ring_pad, outer[1] + ring_pad, outer[2] - ring_pad, outer[3] - ring_pad)
    draw.ellipse(ring, fill=light)
    inner_pad = max(2, int(round((x2 - x1) * 0.24)))
    inner = (outer[0] + inner_pad, outer[1] + inner_pad, outer[2] - inner_pad, outer[3] - inner_pad)
    draw.ellipse(inner, fill=accent)
    core_pad = max(2, int(round((x2 - x1) * 0.34)))
    core = (outer[0] + core_pad, outer[1] + core_pad, outer[2] - core_pad, outer[3] - core_pad)
    draw.ellipse(core, fill=dark)

def finder_module_origins(modules_total, border):
    qr_size_no_border = modules_total - 2 * border
    return [
        (border, border),
        (border + qr_size_no_border - 7, border),
        (border, border + qr_size_no_border - 7),
    ]

def render_halftone_logo_qr(qr, qr_img, size, border, logo_path, mark_shape="plus", density=0.88, logo_strength=0.78, background_dots=True, designer_finders=True):
    if qr is None or not logo_path:
        return None
    dark_rgb = hex_to_rgb(dark_hex)
    brand_rgb = hex_to_rgb(brand_hex)
    light_rgb = hex_to_rgb(light_hex)
    logo_layer, logo_mask = make_logo_mask(logo_path, size)
    canvas = Image.new("RGBA", (size, size), light_rgb + (255,))
    draw = ImageDraw.Draw(canvas)
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total
    finder_origins = finder_module_origins(modules_total, border)
    finder_set = set()
    for fx, fy in finder_origins:
        for yy in range(fy, fy + 7):
            for xx in range(fx, fx + 7):
                finder_set.add((xx, yy))

    for my in range(modules_total):
        for mx in range(modules_total):
            x1 = int(round(mx * module_px)); y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px)); y2 = int(round((my + 1) * module_px))
            if (mx, my) in finder_set:
                continue
            dark = module_is_dark(qr_img, x1, y1, x2, y2)
            inside_logo = bool(logo_mask[y1:y2, x1:x2].mean() > 0.18) if x2 > x1 and y2 > y1 else False
            if not dark and not (background_dots and inside_logo):
                continue
            fallback = brand_rgb + (255,) if inside_logo else dark_rgb + (255,)
            sampled = logo_sample_color(logo_layer, x1, y1, x2, y2, fallback)
            if dark:
                fill = blend_rgb(sampled[:3], dark_rgb, 1.0 - logo_strength) + (255,) if inside_logo else dark_rgb + (255,)
                local_density = density if inside_logo else max(0.54, density * 0.76)
            else:
                fill = blend_rgb(sampled[:3], light_rgb, 0.38) + (185,)
                local_density = max(0.35, density * 0.48)
            draw_halftone_mark(draw, (x1, y1, x2, y2), mark_shape, fill, local_density)

    if designer_finders:
        accents = [brand_rgb + (255,), blend_rgb(brand_rgb, dark_rgb, 0.25) + (255,), brand_rgb + (255,)]
        for (fx, fy), accent in zip(finder_origins, accents):
            x1 = int(round((fx - 1) * module_px)); y1 = int(round((fy - 1) * module_px))
            x2 = int(round((fx + 8) * module_px)); y2 = int(round((fy + 8) * module_px))
            draw_designer_finder(draw, (x1, y1, x2, y2), accent, light_rgb + (255,), dark_rgb + (255,))
    else:
        for fx, fy in finder_origins:
            for yy in range(fy, fy + 7):
                for xx in range(fx, fx + 7):
                    x1 = int(round(xx * module_px)); y1 = int(round(yy * module_px))
                    x2 = int(round((xx + 1) * module_px)); y2 = int(round((yy + 1) * module_px))
                    if module_is_dark(qr_img, x1, y1, x2, y2):
                        draw_module(draw, (x1, y1, x2, y2), "rounded", dark_rgb + (255,))
    return canvas

def visual_quality_metrics(path, logo_path=None):
    img = Image.open(path).convert("RGB")
    gray = np.array(img.convert("L"), dtype=np.float32)
    small = img.resize((256, 256), Image.Resampling.LANCZOS)
    small_gray = np.array(small.convert("L"), dtype=np.float32)
    metrics = {
        "contrast_std": round(float(gray.std()), 2),
        "small_preview_contrast": round(float(small_gray.std()), 2),
    }
    if logo_path:
        try:
            _, mask = make_logo_mask(logo_path, img.size[0])
            metrics["logo_silhouette_coverage"] = round(float(mask.mean()), 3)
        except Exception:
            metrics["logo_silhouette_coverage"] = None
    return metrics

def is_finder_module(mx, my, modules_total, border):
    qr_size_no_border = modules_total - 2 * border
    zones = [
        (border, border),
        (border + qr_size_no_border - 7, border),
        (border, border + qr_size_no_border - 7),
    ]
    for zx, zy in zones:
        if zx <= mx < zx + 7 and zy <= my < zy + 7:
            return True
    return False

def module_color(mx, my, modules_total, mode, dark_rgb, brand_rgb, grad_a, grad_b, artwork_arr=None):
    if mode == "image_based" and artwork_arr is not None:
        h, w = artwork_arr.shape[:2]
        px = min(w - 1, max(0, int(round((mx + 0.5) / modules_total * w))))
        py = min(h - 1, max(0, int(round((my + 0.5) / modules_total * h))))
        sampled = tuple(int(v) for v in artwork_arr[py, px, :3])
        return blend_rgb(sampled, dark_rgb, 0.45) + (255,)
    if mode == "brand_color":
        return brand_rgb + (255,)
    if mode == "gradient":
        t = (mx + my) / max(1, (modules_total - 1) * 2)
        return blend_rgb(grad_a, grad_b, t) + (255,)
    if mode == "two_color":
        t = mx / max(1, modules_total - 1)
        return blend_rgb(dark_rgb, brand_rgb, t) + (255,)
    return dark_rgb + (255,)

def render_styled_qr(qr, qr_img, size, border, module_shape="rounded", finder_style="rounded", color_mode="brand_color", background_style="white", transparent=False, artwork=None):
    dark_rgb = hex_to_rgb(dark_hex)
    brand_rgb = hex_to_rgb(brand_hex)
    grad_a = hex_to_rgb(gradient_start_hex)
    grad_b = hex_to_rgb(gradient_end_hex)
    light_rgb = hex_to_rgb(light_hex)

    canvas = make_background(size, background_style, light_rgb, brand_rgb, artwork=artwork, transparent=transparent)
    draw = ImageDraw.Draw(canvas)
    artwork_arr = np.array(artwork.convert("RGB").resize((size, size), Image.Resampling.LANCZOS)) if artwork is not None else None
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    module_px = size / modules_total

    for my in range(modules_total):
        for mx in range(modules_total):
            x1 = int(round(mx * module_px))
            y1 = int(round(my * module_px))
            x2 = int(round((mx + 1) * module_px))
            y2 = int(round((my + 1) * module_px))
            if not module_is_dark(qr_img, x1, y1, x2, y2):
                continue
            shape = module_shape
            if is_finder_module(mx, my, modules_total, border):
                shape = {"classic": "square", "rounded": "rounded", "circle": "dot"}.get(finder_style, module_shape)
            fill = module_color(mx, my, modules_total, color_mode, dark_rgb, brand_rgb, grad_a, grad_b, artwork_arr=artwork_arr)
            draw_module(draw, (x1, y1, x2, y2), shape, fill)

    return canvas

def apply_logo_treatment(base_img, logo_path, mode="center_badge", size_ratio=0.16, badge_shape="rounded_square", badge_opacity=0.96, badge_padding=0.28, visual_strength=0.90):
    if mode == "none" or not logo_path:
        return base_img.convert("RGBA")

    base = base_img.convert("RGBA")
    size = base.size[0]
    logo_box = max(1, int(round(size * size_ratio)))
    visual_strength = float(np.clip(visual_strength, 0.20, 1.00))
    logo_raw = Image.open(logo_path).convert("RGBA")

    # Rozet/kose modlarinda logo net kalmali; sadece arka plan modlarinda yumusatiladi.
    if mode in ("background_logo", "qr_over_logo_soft"):
        logo = soften_logo_for_qr(logo_raw, visual_strength=visual_strength)
    else:
        rgb = ImageEnhance.Contrast(logo_raw.convert("RGB")).enhance(0.95 + 0.35 * visual_strength)
        rgb = ImageEnhance.Color(rgb).enhance(0.95 + 0.25 * visual_strength)
        logo = rgb.convert("RGBA")
        logo.putalpha(logo_raw.getchannel("A"))

    logo = ImageOps.contain(logo, (logo_box, logo_box), Image.Resampling.LANCZOS)

    if mode == "qr_over_logo_soft":
        bg_logo = ImageOps.contain(logo, (int(size * 0.86), int(size * 0.86)), Image.Resampling.LANCZOS)
        alpha_factor = 0.18 + 0.24 * visual_strength
        alpha = bg_logo.getchannel("A").point(lambda p: int(p * alpha_factor))
        bg_logo.putalpha(alpha)
        layer = Image.new("RGBA", base.size, (255, 255, 255, 0))
        layer.paste(bg_logo, ((size - bg_logo.size[0]) // 2, (size - bg_logo.size[1]) // 2), bg_logo)
        return Image.alpha_composite(layer, base)

    if mode == "background_logo":
        bg_logo = ImageOps.contain(logo, (int(size * 0.72), int(size * 0.72)), Image.Resampling.LANCZOS)
        alpha_factor = 0.26 + 0.34 * visual_strength
        alpha = bg_logo.getchannel("A").point(lambda p: int(p * alpha_factor))
        bg_logo.putalpha(alpha)
        layer = Image.new("RGBA", base.size, (255, 255, 255, 0))
        layer.paste(bg_logo, ((size - bg_logo.size[0]) // 2, (size - bg_logo.size[1]) // 2), bg_logo)
        return Image.alpha_composite(layer, base)

    badge_box = int(round(logo_box * (1.0 + badge_padding * 2.0)))
    layer = Image.new("RGBA", base.size, (255, 255, 255, 0))
    if mode == "corner_small":
        margin = int(size * 0.045)
        bx = size - badge_box - margin
        by = size - badge_box - margin
    else:
        bx = (size - badge_box) // 2
        by = (size - badge_box) // 2

    if mode in ("center_badge", "corner_small", "logo_under_qr"):
        badge = Image.new("RGBA", (badge_box, badge_box), (255, 255, 255, 0))
        bdraw = ImageDraw.Draw(badge)
        fill = (255, 255, 255, int(255 * badge_opacity))
        if badge_shape == "circle":
            bdraw.ellipse((0, 0, badge_box - 1, badge_box - 1), fill=fill)
        elif badge_shape == "square":
            bdraw.rectangle((0, 0, badge_box - 1, badge_box - 1), fill=fill)
        else:
            radius = max(1, int(round(badge_box * 0.18)))
            bdraw.rounded_rectangle((0, 0, badge_box - 1, badge_box - 1), radius=radius, fill=fill)
        layer.paste(badge, (bx, by), badge)
    elif mode == "center_transparent":
        clear = Image.new("RGBA", (badge_box, badge_box), (255, 255, 255, int(255 * 0.45)))
        layer.paste(clear, (bx, by), clear)

    if mode == "corner_small":
        lx = bx + (badge_box - logo.size[0]) // 2
        ly = by + (badge_box - logo.size[1]) // 2
    else:
        lx = (size - logo.size[0]) // 2
        ly = (size - logo.size[1]) // 2
    layer.paste(logo, (lx, ly), logo)
    return Image.alpha_composite(base, layer)


def soften_logo_for_qr(logo, visual_strength=0.58):
    """Logoyu QR ustunde daha az baskin yapar; alfa ve kontrasti dusurur."""
    visual_strength = float(np.clip(visual_strength, 0.20, 1.00))
    logo = logo.convert("RGBA")
    rgb = logo.convert("RGB")
    rgb = ImageEnhance.Contrast(rgb).enhance(0.65 + 0.35 * visual_strength)
    rgb = ImageEnhance.Color(rgb).enhance(0.55 + 0.45 * visual_strength)
    rgb = Image.blend(Image.new("RGB", logo.size, "white"), rgb, visual_strength)
    softened = rgb.convert("RGBA")
    alpha = logo.getchannel("A").point(lambda p: int(p * (0.50 + 0.50 * visual_strength)))
    softened.putalpha(alpha)
    return softened

def logo_clearance_ratio(qr, border):
    """QR yogunlastikca merkezi kapatma alanini kucultur."""
    modules_total = qr.symbol_size(scale=1, border=border)[0]
    data_modules = max(1, modules_total - 2 * border)
    if data_modules <= 29:
        return 0.22
    if data_modules <= 37:
        return 0.18
    return 0.15

def apply_logo_treatment_readable(
    base_img,
    logo_path,
    qr=None,
    border=4,
    expected_data=None,
    mode="center_badge",
    size_ratio=0.16,
    badge_shape="rounded_square",
    badge_opacity=0.96,
    badge_padding=0.28,
    visual_strength=0.90,
    auto_guard=True,
    test_path="qr_fusion_outputs/_logo_readability_probe.png",
):
    if mode == "none" or not logo_path:
        return base_img.convert("RGBA"), {"ok": True, "size_ratio": 0.0, "texts": []}

    capped_ratio = float(size_ratio)
    if qr is not None:
        capped_ratio = min(capped_ratio, logo_clearance_ratio(qr, border))

    ratios = [capped_ratio]
    if auto_guard:
        ratios = sorted(set(round(r, 3) for r in [capped_ratio, capped_ratio * 0.88, capped_ratio * 0.76, capped_ratio * 0.64, capped_ratio * 0.52]), reverse=True)

    best_img = None
    best_result = None
    os.makedirs(os.path.dirname(test_path) or ".", exist_ok=True)

    for ratio in ratios:
        candidate = apply_logo_treatment(
            base_img,
            logo_path,
            mode=mode,
            size_ratio=ratio,
            badge_shape=badge_shape,
            badge_opacity=badge_opacity,
            badge_padding=badge_padding,
            visual_strength=visual_strength,
        )
        save_rgb_or_rgba(candidate, test_path)
        result = test_qr_all(test_path, expected_data=expected_data)
        result["size_ratio"] = ratio
        if best_img is None:
            best_img, best_result = candidate, result
        if result["ok"]:
            return candidate, result

    return best_img, best_result

def save_rgb_or_rgba(img, path):
    if path.lower().endswith((".jpg", ".jpeg")):
        img.convert("RGB").save(path, quality=95)
    else:
        img.save(path)
    return path

def create_social_preview(img, path, canvas_size=1080):
    canvas = Image.new("RGB", (canvas_size, canvas_size), hex_to_rgb(light_hex))
    qr_side = int(canvas_size * 0.82)
    qr_preview = ImageOps.contain(img.convert("RGBA"), (qr_side, qr_side), Image.Resampling.LANCZOS)
    x = (canvas_size - qr_preview.size[0]) // 2
    y = (canvas_size - qr_preview.size[1]) // 2
    canvas.paste(qr_preview, (x, y), qr_preview)
    canvas.save(path, quality=95)
    return path

def write_test_report(report_path, rows):
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("QR Fusion Commercial Test Report\n")
        f.write("================================\n")
        f.write(f"QR data: {qr_data}\n")
        f.write(f"Error level: {error_level}\n")
        f.write(f"Quiet zone: {quiet_zone}\n")
        f.write(f"Module shape: {module_shape}\n")
        f.write(f"Finder style: {finder_style}\n")
        f.write(f"Color mode: {color_mode}\n")
        f.write(f"Logo mode: {logo_mode}\n\n")
        for row in rows:
            f.write(f"{row['path']}\n")
            f.write(f"  ok: {row['result']['ok']}\n")
            f.write(f"  zbar: {row['result']['zbar']}\n")
            f.write(f"  opencv: {row['result']['opencv']}\n")
            if 'visual' in row:
                f.write(f"  visual: {row['visual']}\n")
    return report_path


def create_layout_canvas(qr_img, path, layout="sticker", title=""):
    base = qr_img.convert("RGBA")
    brand = hex_to_rgb(brand_hex)
    light = hex_to_rgb(light_hex)
    if layout == "business_card":
        canvas = Image.new("RGB", (1600, 1000), light)
        qr_side = 560
        qr_small = ImageOps.contain(base, (qr_side, qr_side), Image.Resampling.LANCZOS)
        canvas.paste(qr_small, (940, 220), qr_small)
        draw = ImageDraw.Draw(canvas)
        draw.rectangle((0, 0, 72, 1000), fill=brand)
        draw.text((150, 330), title or brand_text or "SCAN ME", fill=dark_hex)
    elif layout == "label":
        canvas = Image.new("RGB", (1400, 1800), light)
        qr_side = 1040
        qr_small = ImageOps.contain(base, (qr_side, qr_side), Image.Resampling.LANCZOS)
        canvas.paste(qr_small, ((1400 - qr_small.size[0]) // 2, 210), qr_small)
        ImageDraw.Draw(canvas).text((140, 1380), title or brand_text or qr_data, fill=dark_hex)
    else:
        canvas = Image.new("RGB", (1200, 1200), light)
        qr_side = 900
        qr_small = ImageOps.contain(base, (qr_side, qr_side), Image.Resampling.LANCZOS)
        canvas.paste(qr_small, ((1200 - qr_small.size[0]) // 2, 115), qr_small)
        ImageDraw.Draw(canvas).text((150, 1040), title or brand_text or "SCAN", fill=dark_hex)
    canvas.save(path, quality=95)
    return path

def save_segno_vector(qr, svg_path, eps_path=None):
    if qr is None:
        return []
    paths = []
    qr.save(svg_path, kind="svg", border=quiet_zone, dark=dark_hex, light=light_hex)
    paths.append(svg_path)
    if eps_path:
        qr.save(eps_path, kind="eps", border=quiet_zone, dark=dark_hex, light=light_hex)
        paths.append(eps_path)
    return paths


<!-- artqr-doc -->
## 8. Temel Dosyalari Uretme ve Kontrol Etme

Yuklenen gorsel hedef boyuta hazirlanir, temiz QR kod uretilir ve ikisi de ekranda gosterilir. Ardindan temiz QR kod test edilir; temiz QR okunmuyorsa sonraki gorsel birlestirme adimlarina gecmeden hata verilir.


In [ ]:
os.makedirs("qr_fusion_outputs", exist_ok=True)

pad_color = (pad_red, pad_green, pad_blue)
logo_quality = analyze_logo_quality(artwork_path)
print("Logo/artwork kalite analizi:", logo_quality)
for warning in logo_quality["warnings"]:
    print("UYARI:", warning)
prepared_logo_path = prepare_logo_source(artwork_path)
art_img = prepare_artwork(prepared_logo_path, output_size, fit_mode=fit_mode, pad_color=pad_color)

if upload_mode == "use_uploaded_qr_image":
    qr = None
    qr_img = load_uploaded_qr_image(uploaded_qr_path, output_size)
    qr_img.save("qr_fusion_outputs/clean_qr.png")
    qr_designator = "uploaded_qr_image"
else:
    qr, qr_img = make_qr_png(
        qr_data,
        "qr_fusion_outputs/clean_qr.png",
        output_size,
        error=error_level,
        border=quiet_zone,
    )
    qr_designator = qr.designator

art_img.save("qr_fusion_outputs/prepared_artwork.png", quality=95)

print("QR version/designator:", qr_designator)
print("Artwork size:", art_img.size)
display(art_img)
display(qr_img)

clean_test = test_qr_all("qr_fusion_outputs/clean_qr.png", expected_data=expected_data)
print("Temiz QR test:", clean_test)

if not clean_test["ok"]:
    raise ValueError("Temiz QR ticari kabul kuralindan gecmedi. Hazir QR modunda expected_qr_data/qr_data degerini, ya da min_decoder_passes ayarini kontrol et.")


<!-- artqr-doc -->
## 9. Gorsel-QR Adaylarini Uretme

Farkli karisim guclerinde birden fazla aday QR gorseli uretilir. Her aday otomatik olarak test edilir; okunabilen adaylar kaydedilir ve ayara gore ilk basarili sonuc final olarak secilebilir.


In [ ]:
candidate_paths = []
passed_candidates = []
scored_candidates = []

strength_values = np.arange(
    dark_strength_start,
    dark_strength_end + 0.0001,
    dark_strength_step,
)

for idx, strength in enumerate(strength_values, start=1):
    fused = luminance_fusion(
        art_img=art_img,
        qr_img=qr_img,
        qr=qr,
        border=quiet_zone,
        dark_strength=float(strength),
        light_ratio=light_strength_ratio,
        finder_boost=finder_boost,
        quiet_boost=quiet_zone_boost,
        module_contrast=module_contrast,
        center_bias=center_bias,
    )
    fused = polish_image(fused, contrast=contrast_after, sharpness=sharpness_after)

    path = f"qr_fusion_outputs/candidate_{idx:02d}_strength_{strength:.2f}.png"
    fused.save(path, quality=95)
    candidate_paths.append(path)

    result = test_qr_all(path, expected_data=expected_data)
    score = qr_quality_score(path, result, strength=float(strength))
    scored_candidates.append((score, path, float(strength), result))
    print(path, "OK" if result["ok"] else "NO", "score", round(score, 2), result["passed_engines"], result["texts"])

    if show_candidate_previews:
        display(fused.resize((384, 384), Image.Resampling.LANCZOS))

    if result["ok"]:
        passed_candidates.append((path, float(strength), result, score))
        if not save_all_candidates:
            break

if not passed_candidates:
    print("Henuz okunabilir sonuc yok. dark_strength_end, finder_boost, module_contrast veya quiet_zone_boost artir.")
else:
    final_path = "qr_fusion_outputs/final_readable_art_qr.png"
    if prefer_best_score_not_first_pass:
        best_path, best_strength, best_result, best_score = max(passed_candidates, key=lambda item: item[3])
    else:
        best_path, best_strength, best_result, best_score = passed_candidates[0]
    final_img = Image.open(best_path).convert("RGBA")
    logo_path = prepared_logo_path if logo_source_mode == "uploaded_artwork" else None
    final_img, logo_guard = apply_logo_treatment_readable(
        final_img,
        logo_path,
        qr=qr,
        border=quiet_zone,
        expected_data=expected_data,
        mode=logo_mode,
        size_ratio=logo_size_ratio,
        badge_shape=logo_badge_shape,
        badge_opacity=logo_badge_opacity,
        badge_padding=logo_badge_padding,
        visual_strength=logo_visual_strength,
        auto_guard=logo_auto_readability_guard,
        test_path="qr_fusion_outputs/_final_logo_probe.png",
    )
    save_rgb_or_rgba(final_img, final_path)
    final_result = test_qr_all(final_path, expected_data=expected_data)
    print("Final secildi:", final_path)
    print("Strength:", best_strength, "Score:", round(best_score, 2), "Decoder:", final_result["passed_engines"])
    print(f"Logo mode uygulandi: {logo_mode} | ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
    print("Okunan veri:", final_result["texts"])
    display(Image.open(final_path))


<!-- artqr-doc -->
## 10. Adaylari Karsilastirma

Hazirlanan gorsel, temiz QR ve secilen aday tek bir yan yana onizleme icinde gosterilir. Bu hucre, hangi aday dosyasinin hem guzel gorundugunu hem de okunabilir kaldigini hizli kontrol etmek icin kullanilir.


In [ ]:
#@title 10. Aday Inceleme Formu
inspect_candidate_path = "" #@param {type:"string"}

if not inspect_candidate_path.strip():
    if candidate_paths:
        inspect_candidate_path = candidate_paths[-1]
    else:
        raise ValueError("Aday bulunamadi. Once Hucre 7'yi calistir.")

prepared = Image.open("qr_fusion_outputs/prepared_artwork.png").convert("RGB").resize((384, 384), Image.Resampling.LANCZOS)
clean = Image.open("qr_fusion_outputs/clean_qr.png").convert("RGB").resize((384, 384), Image.Resampling.NEAREST)
candidate = Image.open(inspect_candidate_path).convert("RGB").resize((384, 384), Image.Resampling.LANCZOS)

canvas = Image.new("RGB", (384 * 3, 384), "white")
canvas.paste(prepared, (0, 0))
canvas.paste(clean, (384, 0))
canvas.paste(candidate, (384 * 2, 0))

print("Incelenen aday:", inspect_candidate_path)
print(test_qr_all(inspect_candidate_path, expected_data=expected_data))
display(canvas)


<!-- artqr-doc -->
## 11. Manuel Final Secimi

Otomatik secilen sonuc yerine belirli bir aday dosyasini final yapmak isterseniz dosya yolunu burada verebilirsiniz. Secilen dosya once QR testinden gecirilir; okunabiliyorsa final dosyasi olarak kaydedilir.


In [ ]:
#@title 11. Manuel Final Secimi Formu
manual_final_path = "" #@param {type:"string"}

if manual_final_path.strip():
    final_path = "qr_fusion_outputs/final_readable_art_qr.png"
    result = test_qr_all(manual_final_path, expected_data=expected_data)
    if not result["ok"]:
        raise ValueError(f"Secilen aday QR testten gecmedi: {result}")
    Image.open(manual_final_path).save(final_path, quality=95)
    print("Manuel final secildi:", final_path)
    print("Okunan veri:", result["texts"])
    display(Image.open(final_path))
else:
    print("Manuel secim yapilmadi. Hucre 7'nin sectigi final kullanilacak.")


<!-- artqr-doc -->
## 12. Ticari Teslim Paketi Uretme

Bu hucre soft, balanced ve strong gibi farkli tasarim siddetlerinde ticari varyasyonlar uretir. Logo, rozet, renkli modul, saydam PNG, yuksek cozunurluklu PNG/JPG ve paket ZIP dosyasi gibi teslim ciktilari burada hazirlanir.


In [ ]:
delivery_files = []
commercial_rows = []

if commercial_pack_enabled:
    os.makedirs("qr_fusion_outputs/commercial", exist_ok=True)
    logo_path = prepared_logo_path if logo_source_mode == "uploaded_artwork" else None

    if make_soft_balanced_strong:
        commercial_presets = [
            ("soft", max(dark_strength_start, 0.28), 1.05, 1.15),
            ("balanced", max((dark_strength_start + dark_strength_end) / 2, 0.45), contrast_after, sharpness_after),
            ("strong", min(dark_strength_end, 0.90), max(contrast_after, 1.18), max(sharpness_after, 1.45)),
        ]
        for name, strength, contrast_value, sharpness_value in commercial_presets:
            img = luminance_fusion(
                art_img=art_img,
                qr_img=qr_img,
                qr=qr,
                border=quiet_zone,
                dark_strength=float(strength),
                light_ratio=light_strength_ratio,
                finder_boost=finder_boost,
                quiet_boost=quiet_zone_boost,
                module_contrast=module_contrast,
                center_bias=center_bias,
            )
            img = polish_image(img, contrast=contrast_value, sharpness=sharpness_value)
            img, logo_guard = apply_logo_treatment_readable(
                img,
                logo_path,
                qr=qr,
                border=quiet_zone,
                expected_data=expected_data,
                mode=logo_mode,
                size_ratio=logo_size_ratio,
                badge_shape=logo_badge_shape,
                badge_opacity=logo_badge_opacity,
                badge_padding=logo_badge_padding,
                visual_strength=logo_visual_strength,
                auto_guard=logo_auto_readability_guard,
                test_path=f"qr_fusion_outputs/commercial/_logo_probe_{name}.png",
            )
            print(f"Logo guard {name}: ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
            path = f"qr_fusion_outputs/commercial/art_qr_{name}.png"
            save_rgb_or_rgba(img, path)
            result = test_qr_all(path, expected_data=expected_data)
            commercial_rows.append({"path": path, "result": result})
            print(path, "OK" if result["ok"] else "NO", result["texts"])
            if result["ok"]:
                delivery_files.append(path)

    if make_styled_clean_qr and qr is not None:
        styled = render_styled_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            module_shape=module_shape,
            finder_style=finder_style,
            color_mode=color_mode,
            background_style=background_style,
            transparent=False,
            artwork=art_img,
        )
        styled, logo_guard = apply_logo_treatment_readable(
            styled,
            logo_path,
            qr=qr,
            border=quiet_zone,
            expected_data=expected_data,
            mode=logo_mode,
            size_ratio=logo_size_ratio,
            badge_shape=logo_badge_shape,
            badge_opacity=logo_badge_opacity,
            badge_padding=logo_badge_padding,
            visual_strength=logo_visual_strength,
            auto_guard=logo_auto_readability_guard,
            test_path="qr_fusion_outputs/commercial/_logo_probe_styled.png",
        )
        print(f"Logo guard styled: ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
        styled_path = "qr_fusion_outputs/commercial/styled_clean_qr.png"
        save_rgb_or_rgba(styled, styled_path)
        result = test_qr_all(styled_path, expected_data=expected_data)
        commercial_rows.append({"path": styled_path, "result": result})
        print(styled_path, "OK" if result["ok"] else "NO", result["texts"])
        if result["ok"]:
            delivery_files.append(styled_path)

    if make_halftone_logo_qr and qr is not None and logo_path:
        halftone = render_halftone_logo_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            logo_path=logo_path,
            mark_shape=halftone_mark_shape,
            density=halftone_density,
            logo_strength=halftone_logo_strength,
            background_dots=halftone_background_dots,
            designer_finders=halftone_designer_finders,
        )
        halftone_path = "qr_fusion_outputs/commercial/halftone_logo_qr.png"
        save_rgb_or_rgba(halftone, halftone_path)
        result = test_qr_all(halftone_path, expected_data=expected_data)
        visual = visual_quality_metrics(halftone_path, logo_path=logo_path)
        commercial_rows.append({"path": halftone_path, "result": result, "visual": visual})
        print(halftone_path, "OK" if result["ok"] else "NO", result["texts"], "visual", visual)
        if result["ok"]:
            delivery_files.append(halftone_path)

    if make_transparent_png and qr is not None:
        transparent_qr = render_styled_qr(
            qr=qr,
            qr_img=qr_img,
            size=output_size,
            border=quiet_zone,
            module_shape=module_shape,
            finder_style=finder_style,
            color_mode=color_mode,
            background_style="transparent",
            transparent=True,
            artwork=None,
        )
        transparent_qr, logo_guard = apply_logo_treatment_readable(
            transparent_qr,
            logo_path,
            qr=qr,
            border=quiet_zone,
            expected_data=expected_data,
            mode=logo_mode,
            size_ratio=logo_size_ratio,
            badge_shape=logo_badge_shape,
            badge_opacity=logo_badge_opacity,
            badge_padding=logo_badge_padding,
            visual_strength=logo_visual_strength,
            auto_guard=logo_auto_readability_guard,
            test_path="qr_fusion_outputs/commercial/_logo_probe_transparent.png",
        )
        print(f"Logo guard transparent: ratio={logo_guard.get('size_ratio')} ok={logo_guard.get('ok')}")
        transparent_path = "qr_fusion_outputs/commercial/transparent_logo_qr.png"
        save_rgb_or_rgba(transparent_qr, transparent_path)
        # QR okuyucular transparan zemini farkli yorumlayabilir; test icin beyaz zemine kompozitlenir.
        test_canvas = Image.new("RGBA", transparent_qr.size, (255, 255, 255, 255))
        test_canvas.alpha_composite(transparent_qr)
        transparent_test_path = "qr_fusion_outputs/commercial/transparent_logo_qr_white_test.png"
        save_rgb_or_rgba(test_canvas, transparent_test_path)
        result = test_qr_all(transparent_test_path, expected_data=expected_data)
        commercial_rows.append({"path": transparent_path, "result": result})
        print(transparent_path, "OK" if result["ok"] else "NO", result["texts"])
        if result["ok"]:
            delivery_files.append(transparent_path)
            delivery_files.append(transparent_test_path)

    if (make_print_png or make_print_pdf) and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        print_img = Image.open("qr_fusion_outputs/final_readable_art_qr.png").convert("RGB")
        print_img = print_img.resize((4096, 4096), Image.Resampling.LANCZOS)

        if make_print_png:
            print_path = "qr_fusion_outputs/commercial/print_ready_4096.png"
            print_img.save(print_path, quality=95, dpi=(300, 300))
            result = test_qr_all(print_path, expected_data=expected_data)
            commercial_rows.append({"path": print_path, "result": result})
            print(print_path, "OK" if result["ok"] else "NO", result["texts"])
            if result["ok"]:
                delivery_files.append(print_path)

        if make_print_pdf:
            pdf_path = "qr_fusion_outputs/commercial/print_ready_300dpi.pdf"
            print_img.save(pdf_path, "PDF", resolution=300.0)
            delivery_files.append(pdf_path)
            print("Baski PDF hazir:", pdf_path)

    if make_social_preview:
        source_for_social = delivery_files[0] if delivery_files else "qr_fusion_outputs/final_readable_art_qr.png"
        if os.path.exists(source_for_social):
            social_path = "qr_fusion_outputs/commercial/social_preview_1080.png"
            create_social_preview(Image.open(source_for_social), social_path)
            delivery_files.append(social_path)
            print("Sosyal medya onizleme hazir:", social_path)

    if make_styled_clean_qr and qr is None:
        print("Styled clean QR, hazir raster QR modunda atlandi.")

    if make_transparent_png and qr is None:
        print("Transparent styled QR, hazir raster QR modunda atlandi.")

    if make_jpg and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        jpg_path = "qr_fusion_outputs/commercial/final_readable_art_qr.jpg"
        save_rgb_or_rgba(Image.open("qr_fusion_outputs/final_readable_art_qr.png"), jpg_path)
        result = test_qr_all(jpg_path, expected_data=expected_data)
        commercial_rows.append({"path": jpg_path, "result": result})
        if result["ok"]:
            delivery_files.append(jpg_path)
        print(jpg_path, "OK" if result["ok"] else "NO", result["texts"])

    if make_svg or make_eps:
        vector_paths = save_segno_vector(
            qr,
            "qr_fusion_outputs/commercial/clean_qr_vector.svg",
            "qr_fusion_outputs/commercial/clean_qr_vector.eps" if make_eps else None,
        )
        delivery_files.extend(vector_paths)
        if vector_paths:
            print("Vector temiz QR hazir:", vector_paths)
        elif upload_mode == "use_uploaded_qr_image":
            print("SVG/EPS sadece Segno ile uretilen QR icin hazirlanir; hazir raster QR icin atlandi.")

    if make_layout_pack and os.path.exists("qr_fusion_outputs/final_readable_art_qr.png"):
        layout_source = Image.open("qr_fusion_outputs/final_readable_art_qr.png")
        for layout in ["sticker", "business_card", "label"]:
            layout_path = f"qr_fusion_outputs/commercial/{layout}_layout.png"
            create_layout_canvas(layout_source, layout_path, layout=layout, title=brand_text)
            delivery_files.append(layout_path)
            print("Layout hazir:", layout_path)

    if make_test_report:
        report_path = "qr_fusion_outputs/commercial/qr_test_report.txt"
        write_test_report(report_path, commercial_rows)
        delivery_files.append(report_path)
        print("Test raporu hazir:", report_path)

    print("Teslim dosyalari:")
    for path in delivery_files:
        print("-", path)
else:
    delivery_files = []
    print("Ticari paket kapali. Sadece klasik final uretilecek.")


<!-- artqr-doc -->
## 13. Final Dosyayi Bulma, Paketleme ve Indirme

Final QR dosyasi yoksa okunabilir bir yedek dosya aranir ve final olarak kaydedilir. Son olarak cikti klasoru ZIP haline getirilir, dosyalar listelenir ve Colab uzerinden indirme baslatilir.


In [ ]:
final_path = "qr_fusion_outputs/final_readable_art_qr.png"

def find_readable_fallback():
    fallback_paths = []

    for item in globals().get("passed_candidates", []):
        if isinstance(item, (tuple, list)) and item:
            fallback_paths.append(item[0])

    fallback_paths.extend(globals().get("delivery_files", []))
    fallback_paths.extend(globals().get("candidate_paths", []))

    commercial_dir = "qr_fusion_outputs/commercial"
    if os.path.isdir(commercial_dir):
        for name in sorted(os.listdir(commercial_dir)):
            if name.lower().endswith((".png", ".jpg", ".jpeg")):
                fallback_paths.append(os.path.join(commercial_dir, name))

    seen = set()
    for path in fallback_paths:
        if not path or path in seen or not os.path.exists(path):
            continue
        seen.add(path)
        try:
            result = test_qr_all(path, expected_data=expected_data)
        except Exception:
            continue
        if result["ok"]:
            return path, result

    return None, None

if not os.path.exists(final_path):
    fallback_path, fallback_result = find_readable_fallback()
    if fallback_path:
        Image.open(fallback_path).save(final_path, quality=95)
        print("Final dosya yoktu; okunabilir adaydan olusturuldu:", fallback_path)
        print("Okunan veri:", fallback_result["texts"])
    else:
        raise FileNotFoundError(
            "Final dosya bulunamadi ve okunabilir aday da bulunamadi. "
            "Once aday uretim hucrelerini calistir. Adaylar NO cikiyorsa "
            "dark_strength_end=0.95, module_contrast=0.85, finder_boost=2.50, "
            "quiet_zone_boost=1.00 yapip tekrar dene."
        )

result = test_qr_all(final_path, expected_data=expected_data)
print("Final QR test:", result)

if not result["ok"]:
    raise ValueError("Final QR okunamadi. Ayarlari artirip tekrar dene.")

zip_path = "qr_fusion_delivery.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(final_path, arcname="final_readable_art_qr.png")
    z.write("qr_fusion_outputs/clean_qr.png", arcname="clean_qr.png")
    z.write("qr_fusion_outputs/prepared_artwork.png", arcname="prepared_artwork.png")

    for extra_path in globals().get("delivery_files", []):
        if os.path.exists(extra_path):
            z.write(extra_path, arcname=os.path.relpath(extra_path, "qr_fusion_outputs"))

print("Teslim paketi hazir:", zip_path)

if download_final:
    files.download(zip_path)
